# 视频 → SAM 3D Body → 独立 JSON
先完成 README 安装。此 notebook 不下载权重、不安装包、不依赖 Blender。默认单人全画面，正式长视频前先抽取少量帧。

默认并行保存逐帧原始 NPZ（网格/姿态/相机等数值）和轻量 JSON（Blender 交接）。JSON 直接来自同一次模型返回，不是读取 NPZ 再转换。详见 README 的 NPZ 章节。

In [ ]:
from pathlib import Path
import os
import sys

# 在 extractor 目录启动 Jupyter；否则改 EXTRACTOR_DIR
EXTRACTOR_DIR = Path.cwd()
sys.path.insert(0, str(EXTRACTOR_DIR))
VIDEO = Path("input.mp4")
OUTPUT_JSON = Path("motion.json")
SAVE_NPZ = True  # 默认保留 SAM 原始数值输出
NPZ_DIR = Path("motion_npz")  # 必须是新目录，禁止混合多次运行
NPZ_COMPRESSED = True  # False 降低压缩 CPU 开销，但更占空间
SAM_REPO = Path("../sam-3d-body").resolve()
DINOV3_REPO = Path("../dinov3").resolve()
CHECKPOINT = Path("../checkpoints/model.ckpt").resolve()
MHR_MODEL = Path("../checkpoints/assets/mhr_model.pt").resolve()
DEVICE = "auto"  # "cuda" / "cpu" / "auto"
STRIDE = 3  # 每 3 个源帧抽取一个，保留原时间轴
MAX_FRAMES = 2  # 首次验证用 2；正式处理改为 None
START_SECONDS = 0.0
END_SECONDS = None
BBOX = None  # 可设置 [x1,y1,x2,y2] 原画面像素
INFERENCE_TYPE = "body"
os.environ["SAM3D_DINOV3_REPO"] = str(DINOV3_REPO)


## 加载一次，复用模型
CPU 需要应用随附 patch；修改 CPU/CUDA 后重新加载。GPU 是否可用由实际 PyTorch 环境决定。

In [ ]:
from extract_video import load_estimator, extract, validate_document, preflight, preflight_archive
import torch
preflight(VIDEO, OUTPUT_JSON)
if SAVE_NPZ:
    preflight_archive(OUTPUT_JSON, NPZ_DIR)
estimator, runtime_metadata = load_estimator(SAM_REPO, CHECKPOINT, MHR_MODEL, DEVICE)
print(runtime_metadata)


In [ ]:
with torch.inference_mode():
    motion = extract(VIDEO, OUTPUT_JSON, estimator, stride=STRIDE,
                     max_frames=MAX_FRAMES, bbox=BBOX,
                     start_seconds=START_SECONDS, end_seconds=END_SECONDS,
                     inference_type=INFERENCE_TYPE, metadata=runtime_metadata,
                     save_npz=SAVE_NPZ, npz_dir=NPZ_DIR, npz_compressed=NPZ_COMPRESSED)
validate_document(motion)
print("JSON:", OUTPUT_JSON.resolve())
print("samples:", len(motion["frames"]), "source FPS:", motion["fps"])


## 交给第二个工程
只传 OUTPUT_JSON，不需要传模型。源时间轴、米单位、相机坐标和缺帧语义见 README。此输出不是 Blender 骨骼旋转，也不是相机移动/人物走位的世界坐标恢复。

## 检查一个 NPZ
使用 np.load(path, allow_pickle=False)；每次只打开一帧，不要把整段网格读入列表。manifest.json 标记 complete，frames.jsonl 包含源帧号、时间、状态及文件名。失败运行也可能留下 complete=false 的部分归档。

In [ ]:
import numpy as np
if SAVE_NPZ:
    first_npz = next(NPZ_DIR.glob("frame_*.npz"))
    with np.load(first_npz, allow_pickle=False) as raw:
        print({name: (raw[name].shape, str(raw[name].dtype)) for name in raw.files})
